## 3 кейс

**В этом кейсе вы будете рассчитывать:**
* retention
* rolling retention
* lifetime
* churn rate
* mau
* wau
* dau

**Важно**

Перед началом решения задачи выполните следующую ячейку - в ней скачиваются нужные файлы

In [5]:
!wget https://gist.github.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv

!wget https://gist.github.com/Vs8th/aacb80595d1d6aaa2e31eb735f8bc644/raw/entries.csv

!wget https://gist.github.com/Vs8th/0e827e9a608117345dd6585ab81e8c86/raw/metrics.txt

--2026-08-05 04:27:35--  https://gist.github.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv
Resolving gist.github.com (gist.github.com)... 140.82.114.4
Connecting to gist.github.com (gist.github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv [following]
--2026-08-05 04:27:35--  https://gist.githubusercontent.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14918 (15K) [text/plain]
Saving to: ‘registrations.csv.1’

registrations.csv.1 100%[===================>]  14.57K  --.-KB/s    in 0s      

2026-08-05 04:27:35 (103 

Файлами для работы являются `registrations.csv` и `entries.csv`. В них хранятся данные о регистрациях пользователей и входа на платформу соответственно.

### **Посчитайте Retention 15 дня (в процентах) для пользователей, зарегистрированных в январе**

Cохраните результат в переменную `retention_15_day`

**Примечание:** результат округлите до 5 знаков после запятой

In [6]:
# Ваше решение

import csv
from datetime import datetime, timedelta, date
from collections import defaultdict

def parse_date(s):
    return datetime.strptime(s.strip(), "%Y-%m-%d").date()

registrations = {}
with open('registrations.csv', encoding='utf-8') as f:
    for row in csv.DictReader(f, delimiter=';'):
        registrations[row['user_id']] = parse_date(row['registration_date'])

entries = defaultdict(set)
with open('entries.csv', encoding='utf-8') as f:
    for row in csv.DictReader(f, delimiter=';'):
        entries[row['user_id']].add(parse_date(row['entry_date']))

january_cohort = [uid for uid, d in registrations.items() if d.month == 1]

In [8]:
def n_day_retention(users, n):
    if not users:
        return 0.0
    retained = 0
    for uid in users:
        target = registrations[uid] + timedelta(days=n)
        if target in entries.get(uid, set()):
            retained += 1
    return round(retained / len(users) * 100, 5)

retention_15_day = n_day_retention(january_cohort, 15)
print(retention_15_day)

54.65116


In [9]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
# Открываем файл с правильными ответами
with open('metrics.txt', 'r') as f:
    answers = f.read().split('\n')

correct_answer = float(answers[0])

try:
    assert retention_15_day == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Rolling-retention 30 дня (в процентах) для пользователей из той же когорты**

Сохраните результат в переменную `rolling_retention`

**Примечание:** результат округлите до 5 знаков после запятой

In [10]:
# Ваше решение
def n_day_rolling_retention(users, n):
    if not users:
        return 0.0
    retained = 0
    for uid in users:
        target = registrations[uid] + timedelta(days=n)
        if any(d >= target for d in entries.get(uid, set())):
            retained += 1
    return round(retained / len(users) * 100, 5)

rolling_retention = n_day_rolling_retention(january_cohort, 30)
print(rolling_retention)


29.06977


In [11]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[1])

try:
    assert rolling_retention == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Lifetime по всем пользователям, посчитанный как интеграл от n-day retention**

Сохраните результат в переменную `lifetime`

**Примечание:** результат округлите до 5 знаков после запятой

In [12]:
# Ваше решение

def lifetime_metric():
    all_entry_dates = [d for dates in entries.values() for d in dates]
    last_date = max(all_entry_dates)
    min_reg = min(registrations.values())
    max_offset = (last_date - min_reg).days

    total = 0.0
    for n in range(max_offset + 1):
        eligible = [uid for uid, reg in registrations.items()
                    if reg + timedelta(days=n) <= last_date]
        if not eligible:
            continue
        retained = sum(1 for uid in eligible
                        if (registrations[uid] + timedelta(days=n)) in entries.get(uid, set()))
        total += retained / len(eligible)
    return round(total, 5)

lifetime = lifetime_metric()
print(lifetime)

14.83042


In [13]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[2])

try:
    assert lifetime == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Ответы не совпадают


### **Посчитайте Churn rate 29 дня (в долях), посчитанный по всем пользователям**

Сохраните результат в переменную `churn_29`.  
Если будете считать CR от Retention, ведите расчет от Rolling Retention, таким образом, мы получим всех, кто не заходил в 29 день и после. Ведя расчет от обычного Retention, наоборот - получим только CR в 29 день.


In [14]:
# Ваше решение

all_users = list(registrations.keys())

def n_day_rolling_retention_fraction(users, n):
    retained = 0
    for uid in users:
        target = registrations[uid] + timedelta(days=n)
        if any(d >= target for d in entries.get(uid, set())):
            retained += 1
    return retained / len(users)

churn_29 = round(1 - n_day_rolling_retention_fraction(all_users, 29), 5)
print(churn_29)

0.509


In [15]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[3])

try:
    assert churn_29 == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Mau, Wau, Dau за последний месяц/неделю/день записей**

Сохраните результат в переменные `dec_mau`, `dec_wau`, `dec_dau` соответственно

**Примечание:** последний месяц записей - декабрь. Поэтому `mau` рассчитываем для декабря (2021 года), для `wau` берем последнюю неделю - с 25 по 31 декабря, и для `dau` соответственно последний день - 31 декабря.

In [16]:
# Ваше решение
def active_users_count(start, end):
    users = set()
    for uid, dates in entries.items():
        if any(start <= d <= end for d in dates):
            users.add(uid)
    return len(users)

dec_mau = active_users_count(date(2021, 12, 1), date(2021, 12, 31))

print(dec_mau)



133


In [17]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[4])

try:
    assert dec_mau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [18]:
# Ваше решение
def active_users_count(start, end):
    users = set()
    for uid, dates in entries.items():
        if any(start <= d <= end for d in dates):
            users.add(uid)
    return len(users)

dec_wau = active_users_count(date(2021, 12, 25), date(2021, 12, 31))
print(dec_wau)



84


In [19]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[5])

try:
    assert dec_wau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [20]:
# Ваше решение
def active_users_count(start, end):
    users = set()
    for uid, dates in entries.items():
        if any(start <= d <= end for d in dates):
            users.add(uid)
    return len(users)
dec_dau = active_users_count(date(2021, 12, 31), date(2021, 12, 31))
print(dec_dau)


47


In [21]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[6])

try:
    assert dec_dau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Mau, Wau, Dau усредненные**

Сохраните результат в переменные `avg_mau`, `avg_wau`, `avg_dau` соответственно

**Примечание:** результаты округлите до 5 знаков после запятой

In [23]:
# Ваше решение
day_users = defaultdict(set)
week_users = defaultdict(set)
month_users = defaultdict(set)

for uid, dates in entries.items():
    for d in dates:
        day_users[d].add(uid)
        week_users[d.isocalendar()[:2]].add(uid)
        month_users[(d.year, d.month)].add(uid)

avg_dau = round(sum(len(u) for u in day_users.values()) / len(day_users), 5)
avg_wau = round(sum(len(u) for u in week_users.values()) / len(week_users), 5)
avg_mau = round(sum(len(u) for u in month_users.values()) / len(month_users), 5)

In [24]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[7])

try:
    assert avg_mau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [25]:
# Ваше решение
day_users = defaultdict(set)
week_users = defaultdict(set)
month_users = defaultdict(set)

for uid, dates in entries.items():
    for d in dates:
        day_users[d].add(uid)
        week_users[d.isocalendar()[:2]].add(uid)
        month_users[(d.year, d.month)].add(uid)

avg_dau = round(sum(len(u) for u in day_users.values()) / len(day_users), 5)
avg_wau = round(sum(len(u) for u in week_users.values()) / len(week_users), 5)
avg_mau = round(sum(len(u) for u in month_users.values()) / len(month_users), 5)

avg_wau

89.86792

In [26]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[8])

try:
    assert avg_wau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [27]:
# Ваше решение
day_users = defaultdict(set)
week_users = defaultdict(set)
month_users = defaultdict(set)

for uid, dates in entries.items():
    for d in dates:
        day_users[d].add(uid)
        week_users[d.isocalendar()[:2]].add(uid)
        month_users[(d.year, d.month)].add(uid)

avg_dau = round(sum(len(u) for u in day_users.values()) / len(day_users), 5)
avg_wau = round(sum(len(u) for u in week_users.values()) / len(week_users), 5)
avg_mau = round(sum(len(u) for u in month_users.values()) / len(month_users), 5)

avg_dau

40.5589

In [28]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[9])

try:
    assert avg_dau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!
